# Map projection figures

This notebook regenerates figures for the map-projection writeup. The first cell below redraws `Figure1` using only Python, `matplotlib`, and the local `triangulated-land.json` data.

In [ ]:
from pathlib import Path
import json
import math
import os

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Circle, FancyArrowPatch, Polygon, Rectangle

ROOT = Path.cwd()
if ROOT.name != "newFigures":
    root_guess = ROOT / "writ" / "MapProjection" / "newFigures"
    if root_guess.exists():
        ROOT = root_guess
    else:
        raise RuntimeError("Run this notebook from writ/MapProjection/newFigures or the repository root.")

MAP_PROJECTION_DIR = ROOT if ROOT.name == "MapProjection" else ROOT.parent
if ROOT.name == "newFigures":
    MAP_PROJECTION_DIR = ROOT.parent

DATA_DIR = MAP_PROJECTION_DIR / "data"
OUT_DIR = MAP_PROJECTION_DIR / "newFigures"
OUT_DIR.mkdir(exist_ok=True)

# Keeping the matplotlib cache local avoids permissions warnings on this machine.
mpl_cache = OUT_DIR / ".mplcache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

with open(DATA_DIR / "triangulated-land.json") as f:
    land = json.load(f)

pts_groups = [[tuple(pt[:2]) for pt in group] for group in land["pts"]]
tri_groups = land["triangles"]

lon0 = math.radians(-25)
lat0 = math.radians(20)
min_map_lat = math.radians(-55)
max_map_lat = math.radians(83)
merc_min = math.log(math.tan(math.pi / 4 + min_map_lat / 2))
merc_max = math.log(math.tan(math.pi / 4 + max_map_lat / 2))
min_map_lon = math.radians(-178)
max_map_lon = math.radians(178)
min_land_lat = math.radians(-60)

water = "#cfe6f7"
map_bg = "#e8eef3"
land_color = "#8b947f"
grid_color = "#7a8793"
edge_color = "#5f6d77"

fig = plt.figure(figsize=(12, 6.7), dpi=180)
ax = plt.axes([0, 0, 1, 1])
ax.set_xlim(0, 12)
ax.set_ylim(0, 6.7)
ax.set_aspect("equal")
ax.axis("off")

# Overall layout of the composition.
cx, cy, R = 2.7, 3.35, 1.75
mx0, my0, mw, mh = 6.5, 1.2, 5.25, 4.1

def ortho_project(lon, lat, lon0=lon0, lat0=lat0):
    dl = lon - lon0
    cosc = (
        math.sin(lat0) * math.sin(lat)
        + math.cos(lat0) * math.cos(lat) * math.cos(dl)
    )
    x = math.cos(lat) * math.sin(dl)
    y = (
        math.cos(lat0) * math.sin(lat)
        - math.sin(lat0) * math.cos(lat) * math.cos(dl)
    )
    return x, y, cosc

def mercator_xy(lon, lat, lon0=lon0):
    lonr = ((lon - lon0 + math.pi) % (2 * math.pi)) - math.pi
    lonr = max(min_map_lon, min(max_map_lon, lonr))
    latc = max(min_map_lat, min(max_map_lat, lat))
    x = mx0 + (lonr - min_map_lon) / (max_map_lon - min_map_lon) * mw
    y_merc = math.log(math.tan(math.pi / 4 + latc / 2))
    y = my0 + (y_merc - merc_min) / (merc_max - merc_min) * mh
    return x, y

# Globe background.
ax.add_patch(Circle((cx, cy), R, facecolor=water, edgecolor=edge_color, linewidth=1.4, zorder=1))

# A soft highlight keeps the globe from looking too flat.
N = 500
xs = np.linspace(cx - R, cx + R, N)
ys = np.linspace(cy - R, cy + R, N)
X, Y = np.meshgrid(xs, ys)
rx = (X - cx) / R
ry = (Y - cy) / R
mask = rx**2 + ry**2 <= 1
shade = 0.82 + 0.18 * np.clip(1 - 0.5 * rx - 0.9 * ry, 0, 1)
img = np.ones((N, N, 4))
base = np.array([0xCF, 0xE6, 0xF7]) / 255
for i in range(3):
    img[:, :, i] = np.clip(base[i] * shade, 0, 1)
img[:, :, 3] = mask.astype(float)
ax.imshow(
    img,
    extent=[cx - R, cx + R, cy - R, cy + R],
    origin="lower",
    zorder=1.1,
)

# Globe graticule.
for lon_deg in range(-165, 166, 15):
    seg = []
    for lat_deg in np.linspace(-89.5, 89.5, 320):
        x, y, cosc = ortho_project(math.radians(lon_deg), math.radians(lat_deg))
        if cosc >= 0:
            seg.append((cx + R * x, cy + R * y))
        elif len(seg) > 1:
            ax.plot(*zip(*seg), color=grid_color, lw=0.55, alpha=0.6, zorder=2)
            seg = []
    if len(seg) > 1:
        ax.plot(*zip(*seg), color=grid_color, lw=0.55, alpha=0.6, zorder=2)

for lat_deg in range(-75, 76, 15):
    seg = []
    for lon_deg in np.linspace(-180, 180, 500):
        x, y, cosc = ortho_project(math.radians(lon_deg), math.radians(lat_deg))
        if cosc >= 0:
            seg.append((cx + R * x, cy + R * y))
        elif len(seg) > 1:
            ax.plot(*zip(*seg), color=grid_color, lw=0.55, alpha=0.6, zorder=2)
            seg = []
    if len(seg) > 1:
        ax.plot(*zip(*seg), color=grid_color, lw=0.55, alpha=0.6, zorder=2)

# Globe land masses from the triangulated data.
triangles = []
for pts, tri_list in zip(pts_groups, tri_groups):
    proj = [ortho_project(lon, lat) for lon, lat in pts]
    for tri in tri_list:
        verts = [proj[i] for i in tri]
        if any(v[2] < 0 for v in verts):
            continue
        depth = sum(v[2] for v in verts) / 3
        poly = [(cx + R * v[0], cy + R * v[1]) for v in verts]
        triangles.append((depth, poly))

triangles.sort(key=lambda item: item[0])
ax.add_collection(
    PatchCollection(
        [Polygon(poly, closed=True) for _, poly in triangles],
        facecolor=land_color,
        edgecolor="none",
        zorder=3,
    )
)
ax.add_patch(Circle((cx, cy), R, facecolor="none", edgecolor=edge_color, linewidth=1.4, zorder=4))

# Arrow from the globe to the flattened map.
ax.add_patch(
    FancyArrowPatch(
        (5.0, 3.35),
        (6.2, 3.35),
        arrowstyle="Simple,head_length=16,head_width=12,tail_width=1.5",
        color="black",
        lw=0,
        zorder=5,
    )
)

# Mercator panel.
ax.add_patch(Rectangle((mx0, my0), mw, mh, facecolor=map_bg, edgecolor="none", zorder=0.5))

for lon_deg in range(-180, 181, 15):
    x, _ = mercator_xy(math.radians(lon_deg) + lon0, 0)
    ax.plot([x, x], [my0, my0 + mh], color=grid_color, lw=0.8, alpha=0.7, zorder=1)

for lat_deg in range(-75, 76, 15):
    _, y = mercator_xy(0, math.radians(lat_deg))
    ax.plot([mx0, mx0 + mw], [y, y], color=grid_color, lw=0.8, alpha=0.7, zorder=1)

for pts, tri_list in zip(pts_groups, tri_groups):
    for tri in tri_list:
        lons = []
        lats = []
        for i in tri:
            lon, lat = pts[i]
            if lat < min_land_lat:
                lats = []
                break
            lons.append(((lon - lon0 + math.pi) % (2 * math.pi)) - math.pi)
            lats.append(max(min_map_lat, min(max_map_lat, lat)))
        if not lats:
            continue
        if max(lons) - min(lons) > math.pi:
            continue
        poly = [mercator_xy(lon + lon0, lat) for lon, lat in zip(lons, lats)]
        ax.add_patch(Polygon(poly, closed=True, facecolor=land_color, edgecolor="none", zorder=2.5))

ax.add_patch(Rectangle((mx0, my0), mw, mh, facecolor="none", edgecolor=edge_color, linewidth=1.1, zorder=4))

png_path = OUT_DIR / "Figure1.png"
pdf_path = OUT_DIR / "Figure1.pdf"
fig.savefig(png_path, dpi=220, bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"Saved {png_path}")
print(f"Saved {pdf_path}")

## Figure 4

This figure uses two rhumb lines intersecting at a chosen point on the sphere. In the Mercator panel those rhumb lines become straight lines and preserve their angle, while the equirectangular panel distorts that angle.

In [ ]:
fig = plt.figure(figsize=(8, 8), dpi=180)
ax = plt.axes([0, 0, 1, 1])
ax.set_xlim(0, 8)
ax.set_ylim(0, 8)
ax.set_aspect("equal")
ax.axis("off")

cx, cy, R = 4.0, 5.5, 1.7
left_panel = (0.6, 0.6, 2.8, 2.0)
right_panel = (4.6, 0.6, 2.8, 2.0)

edge = "#677785"
grid = "#8796a3"
water = "#d9edf9"
curve = "#111111"

ax.add_patch(Circle((cx, cy), R, facecolor=water, edgecolor=edge, linewidth=1.2))

def ortho4(lon, lat, lon0=math.radians(28), lat0=math.radians(30)):
    dl = lon - lon0
    cosc = math.sin(lat0) * math.sin(lat) + math.cos(lat0) * math.cos(lat) * math.cos(dl)
    x = math.cos(lat) * math.sin(dl)
    y = math.cos(lat0) * math.sin(lat) - math.sin(lat0) * math.cos(lat) * math.cos(dl)
    return x, y, cosc

for lon_deg in range(-180, 181, 10):
    seg = []
    for lat_deg in np.linspace(-89, 89, 300):
        x, y, cosc = ortho4(math.radians(lon_deg), math.radians(lat_deg))
        if cosc >= 0:
            seg.append((cx + R * x, cy + R * y))
        elif len(seg) > 1:
            ax.plot(*zip(*seg), color=grid, lw=0.5, alpha=0.55)
            seg = []
    if len(seg) > 1:
        ax.plot(*zip(*seg), color=grid, lw=0.5, alpha=0.55)

for lat_deg in range(-80, 81, 10):
    seg = []
    for lon_deg in np.linspace(-180, 180, 400):
        x, y, cosc = ortho4(math.radians(lon_deg), math.radians(lat_deg))
        if cosc >= 0:
            seg.append((cx + R * x, cy + R * y))
        elif len(seg) > 1:
            ax.plot(*zip(*seg), color=grid, lw=0.5, alpha=0.55)
            seg = []
    if len(seg) > 1:
        ax.plot(*zip(*seg), color=grid, lw=0.5, alpha=0.55)

for x0, y0, w, h in [left_panel, right_panel]:
    ax.add_patch(Rectangle((x0, y0), w, h, facecolor="white", edgecolor="none"))

min_fig4_lat = math.radians(-60)
max_fig4_lat = math.radians(85)
merc4_min = math.log(math.tan(math.pi / 4 + min_fig4_lat / 2))
merc4_max = math.log(math.tan(math.pi / 4 + max_fig4_lat / 2))
min_fig4_lon = math.radians(-178)
max_fig4_lon = math.radians(178)

def mercator_panel4(lon, lat, panel):
    x0, y0, w, h = panel
    lonr = ((lon + math.pi) % (2 * math.pi)) - math.pi
    lonr = max(min_fig4_lon, min(max_fig4_lon, lonr))
    lat = max(min_fig4_lat, min(max_fig4_lat, lat))
    y = math.log(math.tan(math.pi / 4 + lat / 2))
    X = x0 + (lonr - min_fig4_lon) / (max_fig4_lon - min_fig4_lon) * w
    Y = y0 + (y - merc4_min) / (merc4_max - merc4_min) * h
    return X, Y

def equirect_panel4(lon, lat, panel):
    x0, y0, w, h = panel
    lonr = ((lon + math.pi) % (2 * math.pi)) - math.pi
    lonr = max(min_fig4_lon, min(max_fig4_lon, lonr))
    lat = max(min_fig4_lat, min(max_fig4_lat, lat))
    X = x0 + (lonr - min_fig4_lon) / (max_fig4_lon - min_fig4_lon) * w
    Y = y0 + (lat - min_fig4_lat) / (max_fig4_lat - min_fig4_lat) * h
    return X, Y

for panel, proj in [(left_panel, equirect_panel4), (right_panel, mercator_panel4)]:
    for lon_deg in range(-170, 171, 10):
        xs = []
        ys = []
        for lat_deg in np.linspace(math.degrees(min_fig4_lat), math.degrees(max_fig4_lat), 150):
            X, Y = proj(math.radians(lon_deg), math.radians(lat_deg), panel)
            xs.append(X)
            ys.append(Y)
        ax.plot(xs, ys, color=grid, lw=0.4, alpha=0.55)
    for lat_deg in range(-50, 81, 10):
        xs = []
        ys = []
        for lon_deg in np.linspace(-178, 178, 200):
            X, Y = proj(math.radians(lon_deg), math.radians(lat_deg), panel)
            xs.append(X)
            ys.append(Y)
        ax.plot(xs, ys, color=grid, lw=0.4, alpha=0.55)

path1 = [(math.radians(t), math.radians(90 * math.sin(math.radians(t)))) for t in range(-80, 81)]
path2 = [(math.radians(t - 15), math.radians(90 * math.cos(math.radians(t)))) for t in range(10, 171)]

for path in [path1, path2]:
    lons = []
    lats = []
    for lon, lat in path:
        if min_fig4_lat <= lat <= max_fig4_lat:
            lons.append(lon)
            lats.append(lat)

    xm = []
    ym = []
    xe = []
    ye = []
    for lon, lat in zip(lons, lats):
        X, Y = mercator_panel4(lon, lat, right_panel)
        xm.append(X)
        ym.append(Y)
        X, Y = equirect_panel4(lon, lat, left_panel)
        xe.append(X)
        ye.append(Y)

    ax.plot(xm, ym, color=curve, lw=2.0)
    ax.plot(xe, ye, color=curve, lw=2.0)

    seg = []
    for lon, lat in zip(lons, lats):
        x, y, cosc = ortho4(lon, lat)
        if cosc >= 0:
            seg.append((cx + R * x, cy + R * y))
        elif len(seg) > 1:
            ax.plot(*zip(*seg), color=curve, lw=2.0)
            seg = []
    if len(seg) > 1:
        ax.plot(*zip(*seg), color=curve, lw=2.0)

ax.add_patch(Circle((cx, cy), R, facecolor="none", edgecolor=edge, linewidth=1.2))

png_path = OUT_DIR / "Figure4.png"
pdf_path = OUT_DIR / "Figure4.pdf"
fig.savefig(png_path, dpi=220, bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"Saved {png_path}")
print(f"Saved {pdf_path}")